# Projet : Stage
### Objectif

Pour les attributs spatiaux et temporels, on voudrait faire la même chose. Pour cette partie, il y a 3 étapes à faire : 

1. Identifier les attributs spatiaux / temporels. 
2. Trouver le niveau d'hiérarchie de chaque attributs selon les hiérarchies spatiales / temporelles
3. Identifier le niveau d'hiérarchie le plus fin parmis tous les attributs spatiaux / temporels en tant que granularité minimum de dataset; identifier l'attribut au niveau d'hiérarchie le plus haut parmis tous les attributs en tant que scope de dataset et donner la liste de ses valeurs distinctes. 

L'output final qu'on demande est un dossier json de métadonnée de tous les datasets.

## 1. Hiérarchisation des données

In [ ]:
import os
import re
import json
import csv
import pandas as pd
import load_file as lf
import uml_class as uml

def construire_dictionnaire_hierarchise():

    def fillIris(dic_hierarchise):
        iris_df = pd.read_csv('table_passage_1999_2022.csv', sep=',', encoding='utf-8')
        iris_values = iris_df.dropna().values.astype(str).tolist()
        listeIris = []
        for iris in iris_values:
            for elt in iris:
                if elt not in listeIris and elt != "":
                    listeIris.append(elt.lower())
        
        dic_hierarchise['iris'] = listeIris
        return
    
    def fillCodePostaux(dic_hierarchise):
        code_postaux = pd.read_csv("019HexaSmal.csv", sep=';', encoding='latin-1')
        code_values = code_postaux["Code_postal"]
        code_insee = code_postaux["#Code_commune_INSEE"]
        cleaned_codes = code_values.dropna().astype(str).str.strip().unique().tolist()
        cleaned_insee = code_insee.dropna().astype(str).str.strip().unique().tolist()
        dic_hierarchise["communes"].append(sorted(cleaned_codes))
        dic_hierarchise["communes"].append(sorted(cleaned_insee))

        return

    def fillEPCI(dic_hierarchise):
        df = pd.read_excel("epci-excel.xlsx", dtype=str)
    
        # Nettoyage et concaténation des deux colonnes
        code_epci = df["CODE_EPCI"].dropna().astype(str).str.strip()
        nom_epci = df["NOM_EPCI"].dropna().astype(str).str.strip()

        all_epci = set(code_epci.tolist() + nom_epci.tolist())

        # Fusion avec les valeurs existantes dans le dictionnaire
        existing_epci = set(dic_hierarchise.get("epci", []))
        updated_epci = sorted(existing_epci.union(all_epci))

        dic_hierarchise["epci"] = updated_epci

        return

    def fillChamp(dicChamps, dic_hierarchise, rang):
        for i in range(len(dicChamps['features'])):
            champ = dicChamps['features'][i]['properties']['nom']
            if champ not in dic_hierarchise[rang]:
                dic_hierarchise[rang].append(champ.lower())
        return

    def fillDictionnaireGeoJSON():
        fichiers = ['communes', 'departements', 'regions']
        dic_hierarchise = {}

        for fichier in fichiers:
            dic_hierarchise[fichier] = []
            with open(f"levels/france-geojson/{fichier}-avec-outre-mer.geojson", "r", encoding="utf-8") as mon_json:
                data = json.load(mon_json)
                fillChamp(data, dic_hierarchise, fichier)
        
        dic_hierarchise['regions'].append("ile-de-france")

        return dic_hierarchise

    def fillDictionnaireQuartiers(dic_hierarchise):
        with open("liste-correspondance-qp2024-qp2015.csv", "r", encoding="utf-8") as fichier:
            reader = csv.reader(fichier, delimiter=";")
            listeQuartiers = list(reader)[1:]  # Ignorer l'en-tête

        dic_hierarchise['quartiers'] = []
        for i in range(len(listeQuartiers)):
            quartier = listeQuartiers[i][1]
            if quartier not in dic_hierarchise['quartiers'] and quartier != "":
                dic_hierarchise['quartiers'].append(quartier.lower())
                
        for i in range(len(listeQuartiers)):
            quartier = listeQuartiers[i][3]
            if quartier not in dic_hierarchise['quartiers'] and quartier != "":
                dic_hierarchise['quartiers'].append(quartier.lower())

    dic_hierarchise = fillDictionnaireGeoJSON()
    fillDictionnaireQuartiers(dic_hierarchise)
    fillIris(dic_hierarchise)
    fillEPCI(dic_hierarchise)
    fillCodePostaux(dic_hierarchise)

    # Tri des listes dans le dictionnaire
    champs = ['regions', 'departements', 'communes', 'quartiers', 'QP', 'iris', 'epci']
    dic_hierarchise = {champ: dic_hierarchise[champ] for champ in champs if champ in dic_hierarchise}
    dic_hierarchise['pays'] = ['france', 'france métropolitaine', 'france d\'outre-mer', 'france entière']
    
    return dic_hierarchise

def recuperer_dictionnaire_hierarchise():
    try:
        with open("dic_hierarchise.json", "r", encoding="utf-8") as fichier:
            dic_hierarchise = json.load(fichier)
    except FileNotFoundError:
        dic_hierarchise = construire_dictionnaire_hierarchise()
        with open("dic_hierarchise.json", "w", encoding="utf-8") as fichier:
            json.dump(dic_hierarchise, fichier, ensure_ascii=False, indent=4)
    
    return dic_hierarchise

In [ ]:
# Appel de la fonction pour obtenir le dictionnaire hiérarchisé
dic_hierarchise = recuperer_dictionnaire_hierarchise()
# champs_ranges = ['regions', 'departements', 'communes', 'cantons', 'quartiers', 'QP', 'geopoint']
champs_ranges = ['pays', 'regions', 'departements', 'epci', 'quartiers', 'communes', 'iris', 'geopoints']
temps_ranges = ['annee', 'trimestre', 'mois', 'semaine', 'date']
hierarchie_champs_spa = {champ: (len(champs_ranges) - i) for i, champ in enumerate(champs_ranges)}
hierarchie_temps_spa = {temps: (len(temps_ranges) - i) for i, temps in enumerate(temps_ranges)}

## 2. Identification des attributs spatiaux dans un fichier csv/xlsx

### 2.1 Récupérer tous les attributs spatiaux

#### 2.1.1 Recuperation de tous les datasets

In [ ]:
import os

def getFiles(origine='Opendata'):
    fichiers = []

    for dossier in os.walk(origine):
        for fichier in dossier[2]:
            if fichier.endswith('.csv') or fichier.endswith('.xlsx'):
                fichiers.append(os.path.join(dossier[0], fichier))
    
    return fichiers

datasets = getFiles()

#### 2.1.2 Recherche des attributs spatiaux via contenu des cellules

Variables utiles

In [ ]:
regexAnnee = r'(19\d{2}|20\d{2})$'
regexMois = r'(0[1-9]|1[0-2])'
regexJour = r'(0[1-9]|[12]\d|3[01])'
regexDate = r'(' + regexAnnee[:-1] + r'[-/]?' + regexMois + r'[-/]?' + regexJour + r')'
regexHeure = r'(([10]\d)|(2[0-3]))[:h]([0-5]\d)([:h]([0-5]\d))?'
regexTrim = r'(19\d{2}|20\d{2})_[a-zA-Z]{1}[1-3]'

listeRegexTemporel = [[regexDate, 'date'], [regexAnnee, 'annee'], [regexTrim, 'trimestre']]

Fonctions utiles

In [ ]:
def estGeopoint(cell):
    cell = str(cell).strip()
    match = (
        re.match(r'^(-?\d+(?:\.\d+)?)[,; ]\s*(-?\d+(?:\.\d+)?)$', cell)
        or re.match(r'[0-9]+\s*([a-zA-Z]+\s*[a-zA-Z]+\s)*[0-9]*', cell)
    )
    if match:
        try:
            lat, lon = float(match.group(1)), float(match.group(2))
            if -90 <= lat <= 90 and -180 <= lon <= 180:
                return [True, 'geopoints']
        except Exception:
            pass
    return [False, None]

def estSpatial(cell):
    cell = str(cell).lower()
    infoGeopoint = estGeopoint(cell)
    if infoGeopoint[0]:
        return infoGeopoint
    
    for champ, valeurs in dic_hierarchise.items():
        if cell in valeurs:
            return [True, champ]
        
    return [False, None]

def estTemporel(cell):
    if not isinstance(cell, str):
        cell = str(cell)
    for regex, label in listeRegexTemporel:
        if re.match(regex, cell):
            return [True, label]
    if cell.lower() in ['janvier', 'fevrier', 'mars', 'avril', 'mai', 'juin', 'juillet', 'aout', 'septembre', 'octobre', 'novembre', 'decembre']:
        return [True, 'mois']
    return [False, None]

def recupererAttributsSpatiaux(headers, df, score):
    n_rows = min(10, len(df))
    liste_attributs_spatiaux = {}
    for j, header in enumerate(headers):
        col_values = df[header].astype(str).str.lower().head(n_rows)
        for cell in col_values:
            info = estSpatial(cell)
            if info[0]:
                score[j] += 1
                if score[j] * 10 >= 50 and header not in liste_attributs_spatiaux:
                    liste_attributs_spatiaux[header] = [cell, info[1]]
                    break
    return liste_attributs_spatiaux

def recupererAttributsTemporels(headers, df, score):
    df.dropna(inplace=True)
    n_rows = min(10, len(df))
    liste_attributs_temporels = {}
    for j, header in enumerate(headers):
        col_values = df[header].astype(str).head(n_rows)
        for cell in col_values:
            info = estTemporel(cell)
            if info[0]:
                colonneValide = True
                if info[1] == 'annee':
                    valmin = min(df[header])
                    valmax = max(df[header])
                    if valmin < "1900" or valmax > "2100":
                        colonneValide = False
                        break

                if colonneValide:
                    score[j] += 1
                    if score[j] * 10 >= 50 and header not in liste_attributs_temporels:
                        liste_attributs_temporels[header] = [cell, info[1]]
                        break

    return liste_attributs_temporels

def rechercherLowGranEtScope(liste_attributs, hierarchie):
    if not liste_attributs:
        return {'LowGranularite': [None, None], 'Scope': [None, None]}
    min_att = list(hierarchie.keys())[0]
    max_att = list(hierarchie.keys())[-1]
    le_plus_bas = [min_att, None]
    le_plus_haut = [max_att, None]
    for champ, valeur in liste_attributs.items():
        if hierarchie[valeur[1]] <= hierarchie[le_plus_bas[0]]:
            le_plus_bas = [valeur[1], champ]
        if hierarchie[valeur[1]] >= hierarchie[le_plus_haut[0]]:
            le_plus_haut = [valeur[1], champ]
    result = {'LowGranularite': le_plus_bas, 'Scope': le_plus_haut}
    return result

def chercherEntete(df, max_lignes=20):
    lignes_testees = 0
    old_df = None
    while lignes_testees < max_lignes:
        headers = df.columns.tolist()
        headers_valides = True
        for h in headers:
            if re.match(r'(?i:unnamed|nan)', str(h)):
                headers_valides = False
            if ' ' in str(h).strip():
                headers_valides = False
        if headers_valides:
            for header in headers:
                if (estTemporel(header)[0] or estSpatial(header)[0]) and old_df is not None:
                    df = old_df.copy()
                    break
            return df, False
        if len(df) < 1:
            break
        new_headers = df.iloc[0].tolist()
        old_df = df.copy()
        df = df[1:].copy()
        df.columns = [str(h) for h in new_headers]
        lignes_testees += 1
    return df, True

def spatialScopeToDict(scope, dataset, headers):
    if scope[1] is None:
        return {'spatialScopeLevel': None, 'spatialScope': None}
    
    else :
        scope_level = scope[0]
        scope_values = list(dataset[scope[1]].astype(str).unique())

    return {
        'spatialScopeLevel': scope_level, 
        'spatialScope': scope_values
    }

def temporalScopeToDict(scope, dataset, headers):
    if scope[1] is None:
        return {'temporalScopeLevel': None, 'temporalScopeStart': None, 'temporalScopeEnd': None}
    
    elif scope[1] == 'entêtes':
        scope_level = scope[0]
        scope_values = [col for col in headers if estTemporel(col)[0]]
        
    else:
        scope_level = scope[0]
        scope_values = sorted(dataset[scope[1]].astype(str).unique())

    return {
        'temporalScopeLevel': scope_level,
        'temporalScopeStart': scope_values[0] if scope_values else None,
        'temporalScopeEnd': scope_values[-1] if scope_values else None
    }

def creerDatasetUML(dataset, nom_fichier, extension, granAndScopeSpat, granAndScopeTemp, liste_attributs_spatiaux, liste_attributs_temporels, headers):
    monSpatialScope = uml.DS_Spatial_Scope(None, None).from_dict(spatialScopeToDict(granAndScopeSpat['Scope'], dataset, headers))
    monTemporalScope = uml.DS_Temporal_Scope(None, None, None).from_dict(temporalScopeToDict(granAndScopeTemp['Scope'], dataset, headers))
    title = nom_fichier
    data_content = [uml.Data_Content(champ, valeur[1], 'Spatial') for champ, valeur in liste_attributs_spatiaux.items()]
    data_content += [uml.Data_Content(champ, valeur[1], 'Temporal') for champ, valeur in liste_attributs_temporels.items()]
    monDataset = uml.Dataset(
        title, None, None, extension, None, None, None, None,
        granAndScopeSpat['LowGranularite'][0], monSpatialScope,
        granAndScopeTemp['LowGranularite'][0], monTemporalScope,
        uml.Theme(None, None), data_content
    )
    monDataset.save_to_json(f'metadatas/{nom_fichier}.json')
    return

def process_dataframe(df, nom_fichier, extension, hierarchie_temps_spa, hierarchie_champs_spa):
    headers = df.columns.tolist()
    score_colonne = {k: 0 for k in range(len(headers))}
    df_sample = df.head(10).copy()
    df_sample = df_sample.astype(str)
    liste_attributs_spatiaux = recupererAttributsSpatiaux(headers, df_sample, score_colonne)
    liste_attributs_temporels = recupererAttributsTemporels(headers, df_sample, score_colonne)

    for header in headers:
        if estTemporel(header)[0] and "entêtes" not in liste_attributs_temporels:
            liste_attributs_temporels["entêtes"] = [header, estTemporel(header)[1]]

    low_gran_and_scope_tem = {nom_fichier: rechercherLowGranEtScope(liste_attributs_temporels, hierarchie_temps_spa)}
    low_gran_and_scope_spa = {nom_fichier: rechercherLowGranEtScope(liste_attributs_spatiaux, hierarchie_champs_spa)}

    creerDatasetUML(df, 
                    nom_fichier, 
                    extension, 
                    low_gran_and_scope_spa[nom_fichier], 
                    low_gran_and_scope_tem[nom_fichier], 
                    liste_attributs_spatiaux, 
                    liste_attributs_temporels, 
                    headers)
    return


In [ ]:
# dataset = 'Opendata/Général/Education/base-ic-diplomes-formation-2020_xlsx/base-ic-diplomes-formation-2020.xlsx'
# fichier = dataset.split('/')[-1]
# nom_fichier, extension = fichier.split('.')

# try:
#     listeSheets = pd.ExcelFile(dataset)
#     nbSheets = len(listeSheets.sheet_names)
#     for i in range(nbSheets):
#         try:
#             df = lf.find_type(dataset, i)[0]
#             df, feuilleInvalides = chercherEntete(df)
#             if feuilleInvalides:
#                 print(f"Feuille {i} invalide, passage à la suivante.")
            
#             process_dataframe(df, 
#                               f"{nom_fichier}_sheet{i}", 
#                               extension,
#                               hierarchie_temps_spa, 
#                               hierarchie_champs_spa)
            
#         except Exception as e:
#             print(f"Erreur lors de la lecture de la feuille {i} : {e}")
# except Exception as e:
#     print(f"Erreur générale sur {nom_fichier}: {e}")

In [ ]:
for dataset in datasets:
    fichier = dataset.split('/')[-1]
    nom_fichier, extension = fichier.split('.')
    print(f"Traitement du fichier {nom_fichier}")

    try:
        if extension == 'xlsx':
            listeSheets = pd.ExcelFile(dataset)
            nbSheets = len(listeSheets.sheet_names)
            for i in range(nbSheets):
                try:
                    df = lf.find_type(dataset, i)[0]
                    df, feuilleInvalides = chercherEntete(df)
                    if feuilleInvalides:
                        print(f"\t- Feuille {i} invalide, passage à la suivante.")
                        continue
                    
                    process_dataframe(df, 
                                      f"{nom_fichier}_sheet{i}", 
                                      extension,
                                      hierarchie_temps_spa, 
                                      hierarchie_champs_spa)
                    
                except Exception as e:
                    print(f"\t  /!\\ Erreur lors de la lecture de la feuille {i} : {e}")
                    continue
        elif extension == 'csv':
            try:
                df = lf.find_type(dataset)[0]
                if len(df.columns) < 5:
                    try:
                        df = pd.read_csv(dataset, sep=';', encoding='utf-8')
                    except:
                        df = pd.read_csv(dataset, sep=';', encoding='latin1')

                process_dataframe(df, 
                                  nom_fichier, 
                                  extension,
                                  hierarchie_temps_spa, 
                                  hierarchie_champs_spa)
                
            except Exception as e:
                print(f"Erreur lors du chargement du fichier {nom_fichier[:10]}: {e}")
                continue
        else:
            print(f"Extension non supportée pour {nom_fichier}")
            continue
    except Exception as e:
        print(f"Erreur générale sur {nom_fichier}: {e}")
        continue

In [ ]:
del champs_ranges, dataset, datasets, df, dic_hierarchise, extension, feuilleInvalides, fichier, hierarchie_champs_spa, hierarchie_temps_spa,i, listeRegexTemporel, listeSheets, nbSheets, nom_fichier, regexAnnee, regexDate, regexHeure, regexJour, regexMois, regexTrim, temps_ranges